# Pan-Cancer Overlap Markers: KM Curves (Adjusted vs Unadjusted)

For each pan-cancer marker that overlaps across specifications (from `pan_cancer_overlap_analysis.ipynb`):

**Track 1 (ICI-only)**: 2 columns (ATE-weighted vs unweighted) x rows per cohort
**Track 2 (Interaction)**: 4 columns (ATE ICI, ATE non-ICI, noIPTW ICI, noIPTW non-ICI) x rows per cohort

All weighting mirrors `run_IPTW_analysis.py`:
- Stabilized ATE weights with common support trimming at (0.5, 99.5) and truncation at (1, 99)
- Generalizability weights for Track 1 ICI-only: 1/ps
- Bootstrap 95% CI for weighted curves, Greenwood CI for unweighted

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from lifelines.exceptions import StatisticalWarning
from sklearn.linear_model import LogisticRegression

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "figure.dpi": 150,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
})

COLOR_POS = "#D62728"
COLOR_NEG = "#1F77B4"
CI_ALPHA = 0.18

# ── Paths ─────────────────────────────────────────────────────────────
DATA_PATH = '/data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/'
MARKER_PATH = os.path.join(DATA_PATH, 'biomarker_analysis/')
COMPILED_DIR = os.path.join(MARKER_PATH, 'compiled_results/')
FIGURE_PATH = '/data/gusev/USERS/jpconnor/figures/clinical_text_embedding_project/'
KM_FIG_PATH = os.path.join(FIGURE_PATH, 'biomarker_analysis/pan_cancer_overlap_KM/')
os.makedirs(KM_FIG_PATH, exist_ok=True)

# ── Configuration ─────────────────────────────────────────────────────
COHORTS = ['cohort1', 'cohort2']
PS_MODELS = ['covariates_only', 'covariates_plus_embeddings']
N_BOOT = 250
GRID_POINTS = 200
SEED = 42
MAX_TIME_DAYS = None

# ── Load overlap markers ──────────────────────────────────────────────
overlap_df = pd.read_csv(os.path.join(COMPILED_DIR, 'pan_cancer_overlap_markers_for_km.csv'))
OVERLAP_MARKERS = sorted(overlap_df['marker'].tolist())
print(f"Loaded {len(OVERLAP_MARKERS)} pan-cancer overlap markers:")
for m in OVERLAP_MARKERS:
    print(f"  {m}")

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────

COMMON_SUPPORT_PCT = (0.5, 99.5)
IPTW_TRUNC_PCT = (1, 99)
EPS = 1e-6


def recalibrate_propensity_within_subset(df, ps_col="ICI_prediction", treat_col="PX_on_ICI"):
    ps = df[[ps_col]].values
    y = df[treat_col].values.astype(int)
    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=1000)
    lr.fit(ps, y)
    return lr.predict_proba(ps)[:, 1]


def prepare_pan_cancer_weighted_df(full_df):
    """Prepare pan-cancer weighted DataFrame mirroring run_IPTW_analysis.py."""
    df = full_df.copy()

    ps_raw = df["ICI_prediction"].clip(EPS, 1 - EPS)
    ps_t = ps_raw[df["PX_on_ICI"] == 1]
    ps_c = ps_raw[df["PX_on_ICI"] == 0]

    lower = max(np.percentile(ps_t, COMMON_SUPPORT_PCT[0]),
                np.percentile(ps_c, COMMON_SUPPORT_PCT[0]))
    upper = min(np.percentile(ps_t, COMMON_SUPPORT_PCT[1]),
                np.percentile(ps_c, COMMON_SUPPORT_PCT[1]))
    df = df[(ps_raw >= lower) & (ps_raw <= upper)].copy()

    ps = df["ICI_prediction"].clip(EPS, 1 - EPS)
    treat = df["PX_on_ICI"]
    p_treated = treat.mean()

    # Stabilized ATE weights (Track 2)
    w_ate = np.where(treat == 1, p_treated / ps, (1 - p_treated) / (1 - ps))
    lo, hi = np.percentile(w_ate, IPTW_TRUNC_PCT)
    df["IPTW_ATE"] = np.clip(w_ate, lo, hi)

    # Generalizability weights for ICI-only (Track 1): w = 1/ps
    ici_mask = treat == 1
    ici_ps = ps[ici_mask]
    w_gen = 1.0 / ici_ps
    lo_gen, hi_gen = np.percentile(w_gen, IPTW_TRUNC_PCT)
    if np.isfinite(lo_gen) and np.isfinite(hi_gen):
        df.loc[ici_mask, "IPTW_GEN"] = np.clip(w_gen, lo_gen, hi_gen).values
    else:
        df.loc[ici_mask, "IPTW_GEN"] = 1.0

    return df


def bootstrap_weighted_km(durations, events, weights, grid,
                          n_boot=250, seed=42, renormalize=True):
    n = len(durations)
    if n == 0:
        return np.ones(len(grid)), np.ones(len(grid)), np.ones(len(grid))

    dur = np.asarray(durations)
    evt = np.asarray(events)
    wgt = np.asarray(weights, dtype=float)

    if renormalize:
        s = wgt.sum()
        if s > 0:
            wgt = wgt * (n / s)

    rng = np.random.default_rng(seed)
    km = KaplanMeierFitter()
    surv_mat = np.empty((n_boot, len(grid)))

    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore", category=StatisticalWarning)
            km.fit(dur[idx], evt[idx], weights=wgt[idx])
        surv_mat[b, :] = km.survival_function_at_times(grid).values

    return (np.median(surv_mat, axis=0),
            np.quantile(surv_mat, 0.025, axis=0),
            np.quantile(surv_mat, 0.975, axis=0))


def unweighted_km(durations, events, grid):
    km = KaplanMeierFitter()
    km.fit(durations, events)
    surv = km.survival_function_at_times(grid).values
    ci = km.confidence_interval_survival_function_
    ci_lo = np.interp(grid, ci.index, ci.iloc[:, 0])
    ci_hi = np.interp(grid, ci.index, ci.iloc[:, 1])
    return surv, ci_lo, ci_hi


def _format_ax(ax, title, xlabel="Time (days)", ylabel="Survival probability"):
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_ylim(-0.02, 1.02)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def _p_str(p):
    if p is None:
        return ""
    return f"p={p:.2e}" if p < 0.001 else f"p={p:.3f}"


print("Helper functions defined.")

In [ ]:
# ── Load IPTW DataFrames and prepare weighted data ────────────────────

# weighted_cache[(cohort, ps_model)] -> weighted pan-cancer DataFrame
weighted_cache = {}

for cohort in COHORTS:
    for ps_model in PS_MODELS:
        spec_label = f"{cohort}_{ps_model}"
        iptw_file = os.path.join(MARKER_PATH, f"IPTW_df_{spec_label}.csv")
        print(f"\nLoading {spec_label}...", end=" ")

        try:
            spec_df = pd.read_csv(iptw_file)
        except FileNotFoundError:
            print("FILE NOT FOUND — skipping")
            continue

        wdf = prepare_pan_cancer_weighted_df(spec_df)
        weighted_cache[(cohort, ps_model)] = wdf

        n_ici = int((wdf["PX_on_ICI"] == 1).sum())
        n_ctrl = len(wdf) - n_ici
        print(f"{len(wdf)} patients ({n_ici} ICI, {n_ctrl} non-ICI)")

print(f"\nCached {len(weighted_cache)} (cohort, ps_model) combinations")

## Track 1: ATE-Weighted vs Unweighted KM (ICI-only)

For each pan-cancer overlap marker: rows = (cohort, ps_model), columns = (ATE-weighted, unweighted).
One figure per marker showing all available specifications.

In [ ]:
# ── Track 1 KM panel helper ───────────────────────────────────────────

def _plot_km_panel(ax, d_pos, d_neg, grid, weighted, weight_col=None):
    """Plot a single KM panel. Returns log-rank p-value or None."""
    if len(d_pos) < 5 or len(d_neg) < 5:
        ax.text(0.5, 0.5, f"n+ = {len(d_pos)}, n- = {len(d_neg)}\n(insufficient)",
                ha='center', va='center', transform=ax.transAxes)
        return None

    if weighted and weight_col is not None:
        lr = logrank_test(
            d_pos["tt_death"], d_neg["tt_death"],
            event_observed_A=d_pos["death"], event_observed_B=d_neg["death"],
            weights_A=d_pos[weight_col], weights_B=d_neg[weight_col])

        med_p, lo_p, hi_p = bootstrap_weighted_km(
            d_pos["tt_death"], d_pos["death"], d_pos[weight_col], grid, N_BOOT, SEED)
        med_n, lo_n, hi_n = bootstrap_weighted_km(
            d_neg["tt_death"], d_neg["death"], d_neg[weight_col], grid, N_BOOT, SEED + 1)

        ax.plot(grid, med_p, color=COLOR_POS, lw=2, label=f"Mut+ (n={len(d_pos)})")
        ax.fill_between(grid, lo_p, hi_p, color=COLOR_POS, alpha=CI_ALPHA)
        ax.plot(grid, med_n, color=COLOR_NEG, lw=2, label=f"Mut- (n={len(d_neg)})")
        ax.fill_between(grid, lo_n, hi_n, color=COLOR_NEG, alpha=CI_ALPHA)
    else:
        lr = logrank_test(
            d_pos["tt_death"], d_neg["tt_death"],
            event_observed_A=d_pos["death"], event_observed_B=d_neg["death"])

        s_p, lo_p, hi_p = unweighted_km(d_pos["tt_death"], d_pos["death"], grid)
        s_n, lo_n, hi_n = unweighted_km(d_neg["tt_death"], d_neg["death"], grid)

        ax.plot(grid, s_p, color=COLOR_POS, lw=2, label=f"Mut+ (n={len(d_pos)})")
        ax.fill_between(grid, lo_p, hi_p, color=COLOR_POS, alpha=CI_ALPHA)
        ax.plot(grid, s_n, color=COLOR_NEG, lw=2, label=f"Mut- (n={len(d_neg)})")
        ax.fill_between(grid, lo_n, hi_n, color=COLOR_NEG, alpha=CI_ALPHA)

    ax.legend(fontsize=8, loc="lower left")
    return lr.p_value


def _binarize_marker(df, marker):
    """Binarize marker column and split into pos/neg."""
    d = df[["tt_death", "death", marker]].copy()
    if "IPTW_GEN" in df.columns:
        d["IPTW_GEN"] = df["IPTW_GEN"]
    if "IPTW_ATE" in df.columns:
        d["IPTW_ATE"] = df["IPTW_ATE"]
    d = d.dropna(subset=["tt_death", "death", marker])
    d[marker] = (pd.to_numeric(d[marker], errors="coerce").fillna(0) > 0).astype(int)
    return d[d[marker] == 1], d[d[marker] == 0]


print("Panel helpers defined.")

In [ ]:
# ── Track 1: ATE vs Unweighted KM for each marker ────────────────────

COHORT_LABELS = {'cohort1': 'Unmatched', 'cohort2': 'Matched (1:1)'}
PS_LABELS = {'covariates_only': 'Cov-only PS',
             'covariates_plus_embeddings': 'Cov+Emb PS'}

t1_pvals = []

for marker in OVERLAP_MARKERS:
    # Determine which specs have this marker column
    specs = [(c, p) for (c, p) in weighted_cache if marker in weighted_cache[(c, p)].columns]
    if not specs:
        print(f"  Skip {marker}: not in any IPTW DataFrame")
        continue

    n_rows = len(specs)
    fig, axes = plt.subplots(n_rows, 2, figsize=(14, 5.5 * n_rows), squeeze=False, sharey=True)

    for row_i, (cohort, ps_model) in enumerate(sorted(specs)):
        df = weighted_cache[(cohort, ps_model)]
        ici_df = df[df["PX_on_ICI"] == 1].copy()

        d_pos, d_neg = _binarize_marker(ici_df, marker)

        tmax = float(ici_df["tt_death"].max()) if len(ici_df) else 1.0
        if MAX_TIME_DAYS is not None:
            tmax = min(tmax, MAX_TIME_DAYS)
        grid = np.linspace(0, tmax, GRID_POINTS)

        # ATE-weighted (left)
        p_w = _plot_km_panel(axes[row_i, 0], d_pos, d_neg, grid,
                             weighted=True, weight_col="IPTW_GEN")
        _format_ax(axes[row_i, 0], f"ATE-weighted\n{_p_str(p_w)}")

        # Unweighted (right)
        p_u = _plot_km_panel(axes[row_i, 1], d_pos, d_neg, grid,
                             weighted=False)
        _format_ax(axes[row_i, 1], f"Unweighted\n{_p_str(p_u)}")

        axes[row_i, 0].set_ylabel(
            f"{COHORT_LABELS.get(cohort, cohort)}\n{PS_LABELS.get(ps_model, ps_model)}\nSurvival probability")

        t1_pvals.append({
            'marker': marker, 'cohort': cohort, 'ps_model': ps_model,
            'p_weighted': p_w, 'p_unweighted': p_u,
            'n_pos': len(d_pos), 'n_neg': len(d_neg),
        })

    fig.suptitle(f"{marker} — Pan-Cancer\nTrack 1: ICI-only (ATE vs Unweighted)",
                 fontweight="bold", y=1.02)
    fig.tight_layout()
    fig.savefig(os.path.join(KM_FIG_PATH, f"T1_{marker}.png"))
    plt.show()

t1_pvals_df = pd.DataFrame(t1_pvals)
print(f"\nCollected {len(t1_pvals_df)} Track 1 p-value comparisons")

## Track 2: ATE-Weighted vs noIPTW KM (ICI & Non-ICI Arms)

For each pan-cancer overlap marker: 4 columns (ATE ICI, ATE non-ICI, noIPTW ICI, noIPTW non-ICI) x rows per (cohort, ps_model).

In [ ]:
# ── Track 2: ATE vs noIPTW KM for each marker ────────────────────────

t2_pvals = []

for marker in OVERLAP_MARKERS:
    specs = [(c, p) for (c, p) in weighted_cache if marker in weighted_cache[(c, p)].columns]
    if not specs:
        continue

    n_rows = len(specs)
    fig, axes = plt.subplots(n_rows, 4, figsize=(26, 5.5 * n_rows), squeeze=False, sharey=True)

    for row_i, (cohort, ps_model) in enumerate(sorted(specs)):
        df = weighted_cache[(cohort, ps_model)]
        ici_df = df[df["PX_on_ICI"] == 1].copy()
        nonici_df = df[df["PX_on_ICI"] == 0].copy()

        tmax = float(df["tt_death"].max()) if len(df) else 1.0
        if MAX_TIME_DAYS is not None:
            tmax = min(tmax, MAX_TIME_DAYS)
        grid = np.linspace(0, tmax, GRID_POINTS)

        # ICI arm
        ici_pos, ici_neg = _binarize_marker(ici_df, marker)
        # Non-ICI arm
        ni_pos, ni_neg = _binarize_marker(nonici_df, marker)

        # Col 0: ATE-weighted ICI
        p = _plot_km_panel(axes[row_i, 0], ici_pos, ici_neg, grid,
                           weighted=True, weight_col="IPTW_ATE")
        _format_ax(axes[row_i, 0], f"ICI — ATE-weighted\n{_p_str(p)}")

        # Col 1: ATE-weighted non-ICI
        p2 = _plot_km_panel(axes[row_i, 1], ni_pos, ni_neg, grid,
                            weighted=True, weight_col="IPTW_ATE")
        _format_ax(axes[row_i, 1], f"Non-ICI — ATE-weighted\n{_p_str(p2)}")

        # Col 2: noIPTW ICI
        p3 = _plot_km_panel(axes[row_i, 2], ici_pos, ici_neg, grid,
                            weighted=False)
        _format_ax(axes[row_i, 2], f"ICI — noIPTW\n{_p_str(p3)}")

        # Col 3: noIPTW non-ICI
        p4 = _plot_km_panel(axes[row_i, 3], ni_pos, ni_neg, grid,
                            weighted=False)
        _format_ax(axes[row_i, 3], f"Non-ICI — noIPTW\n{_p_str(p4)}")

        axes[row_i, 0].set_ylabel(
            f"{COHORT_LABELS.get(cohort, cohort)}\n{PS_LABELS.get(ps_model, ps_model)}\n"
            f"Survival probability")

        t2_pvals.append({
            'marker': marker, 'cohort': cohort, 'ps_model': ps_model,
            'p_ATE_ICI': p, 'p_ATE_nonICI': p2,
            'p_noIPTW_ICI': p3, 'p_noIPTW_nonICI': p4,
            'n_ici_pos': len(ici_pos), 'n_ici_neg': len(ici_neg),
            'n_nonici_pos': len(ni_pos), 'n_nonici_neg': len(ni_neg),
        })

    fig.suptitle(f"{marker} — Pan-Cancer\nTrack 2: ICI vs Non-ICI x ATE vs noIPTW",
                 fontweight="bold", y=1.02)
    fig.tight_layout()
    fig.savefig(os.path.join(KM_FIG_PATH, f"T2_{marker}.png"))
    plt.show()

t2_pvals_df = pd.DataFrame(t2_pvals)
print(f"\nCollected {len(t2_pvals_df)} Track 2 p-value comparisons")

## Summary: Weighted vs Unweighted p-value Comparison

In [ ]:
# ── p-value scatter: weighted vs unweighted ───────────────────────────

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Track 1
if len(t1_pvals_df) > 0:
    valid = t1_pvals_df.dropna(subset=['p_weighted', 'p_unweighted'])
    ax1.scatter(-np.log10(valid['p_unweighted']),
                -np.log10(valid['p_weighted']),
                c=valid['cohort'].map({'cohort1': '#1f77b4', 'cohort2': '#d62728'}),
                s=40, alpha=0.7, edgecolors='k', linewidth=0.3)

    lims = [0, max(ax1.get_xlim()[1], ax1.get_ylim()[1])]
    ax1.plot(lims, lims, 'k--', alpha=0.3, lw=1)
    ax1.axhline(-np.log10(0.05), color='grey', ls=':', lw=0.8, alpha=0.5)
    ax1.axvline(-np.log10(0.05), color='grey', ls=':', lw=0.8, alpha=0.5)
    ax1.set_xlabel('-log10(p) Unweighted')
    ax1.set_ylabel('-log10(p) ATE-weighted')
    ax1.set_title('Track 1: ICI-only log-rank', fontweight='bold')
    ax1.legend(handles=[
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#1f77b4', label='cohort1'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#d62728', label='cohort2'),
    ], loc='upper left')

# Track 2 (ICI arm)
if len(t2_pvals_df) > 0:
    valid2 = t2_pvals_df.dropna(subset=['p_ATE_ICI', 'p_noIPTW_ICI'])
    ax2.scatter(-np.log10(valid2['p_noIPTW_ICI']),
                -np.log10(valid2['p_ATE_ICI']),
                c=valid2['cohort'].map({'cohort1': '#1f77b4', 'cohort2': '#d62728'}),
                s=40, alpha=0.7, edgecolors='k', linewidth=0.3)

    lims = [0, max(ax2.get_xlim()[1], ax2.get_ylim()[1])]
    ax2.plot(lims, lims, 'k--', alpha=0.3, lw=1)
    ax2.axhline(-np.log10(0.05), color='grey', ls=':', lw=0.8, alpha=0.5)
    ax2.axvline(-np.log10(0.05), color='grey', ls=':', lw=0.8, alpha=0.5)
    ax2.set_xlabel('-log10(p) noIPTW')
    ax2.set_ylabel('-log10(p) ATE-weighted')
    ax2.set_title('Track 2: ICI arm log-rank', fontweight='bold')
    ax2.legend(handles=[
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#1f77b4', label='cohort1'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#d62728', label='cohort2'),
    ], loc='upper left')

fig.tight_layout()
fig.savefig(os.path.join(KM_FIG_PATH, 'pvalue_weighted_vs_unweighted.png'))
plt.show()

# Save p-value tables
t1_pvals_df.to_csv(os.path.join(KM_FIG_PATH, 'T1_km_pvalue_comparison.csv'), index=False)
t2_pvals_df.to_csv(os.path.join(KM_FIG_PATH, 'T2_km_pvalue_comparison.csv'), index=False)
print(f"Saved p-value CSVs to {KM_FIG_PATH}")